In [ ]:
import os

# Mount Google Drive and set working directory
from google.colab import drive
drive.mount("/content/drive")

os.chdir("/content/drive/MyDrive/Colab Notebooks/ml/notebooks")
print("Working directory:", os.getcwd())

# Phase 4 — Neural Collaborative Filtering (NCF)

**Lead: Yunus** | Steps 4.1 – 4.5

| Step | Description |
|---|---|
| 4.1 | Define NCF model: Embedding(user) + Embedding(product) → MLP[256,128,64] → Sigmoid |
| 4.2 | Build dataset with negative sampling (ratio 1:1) |
| 4.3 | Train: Adam (lr=1e-3), BCE loss, batch=4096, early stopping (patience=3) |
| 4.4 | Evaluate: Precision, Recall, F1, ROC-AUC |
| 4.5 | t-SNE visualization of product embeddings colored by department |

**Reference:** He et al., 2017. Neural Collaborative Filtering. WWW '17.

**Prerequisite:** Run `02_feature_engineering.ipynb` first.


In [ ]:
import os
import sys
import json
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.manifold import TSNE
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

sys.path.insert(0, '..')
from models.ncf import NCFModel, NCFDataset

FEATURES_DIR = '../features'
MODELS_DIR   = '../saved_models'
OUTPUTS_DIR  = '../outputs'

os.makedirs(MODELS_DIR, exist_ok=True)
sns.set_style('darkgrid')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
feature_df = pd.read_parquet(f'{FEATURES_DIR}/feature_matrix.parquet')
prior      = pd.read_parquet(f'{FEATURES_DIR}/prior_merged.parquet')

print(f'feature_df: {feature_df.shape}')
print(f'prior:      {prior.shape}')

## Step 4.1 — NCF Model Architecture

In [ ]:
# Build integer index maps for users and products
all_user_ids    = feature_df['user_id'].unique()
all_product_ids = feature_df['product_id'].unique()

user2idx    = {u: i for i, u in enumerate(all_user_ids)}
product2idx = {p: i for i, p in enumerate(all_product_ids)}

n_users    = len(user2idx)
n_products = len(product2idx)
print(f'Unique users: {n_users:,}  |  Unique products: {n_products:,}')

feature_df['user_idx']    = feature_df['user_id'].map(user2idx)
feature_df['product_idx'] = feature_df['product_id'].map(product2idx)

# NCF model: embed_dim=64, MLP [256, 128, 64], dropout=0.3
model = NCFModel(
    n_users=n_users,
    n_products=n_products,
    embed_dim=64,
    mlp_layers=[256, 128, 64],
    dropout=0.3,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')
print(model)

## Step 4.2 — Dataset with Negative Sampling (1:4 ratio)

In [ ]:
train_df = feature_df[feature_df['split'] == 'train']
val_df   = feature_df[feature_df['split'] == 'val']
test_df  = feature_df[feature_df['split'] == 'test']

print('Building datasets with negative sampling...')

train_dataset = NCFDataset(
    user_ids=train_df['user_idx'].values,
    product_ids=train_df['product_idx'].values,
    labels=train_df['reordered'].values,
    n_products=n_products,
    neg_ratio=1,
)
val_dataset = NCFDataset(
    user_ids=val_df['user_idx'].values,
    product_ids=val_df['product_idx'].values,
    labels=val_df['reordered'].values,
    n_products=n_products,
    neg_ratio=0,
)
test_dataset = NCFDataset(
    user_ids=test_df['user_idx'].values,
    product_ids=test_df['product_idx'].values,
    labels=test_df['reordered'].values,
    n_products=n_products,
    neg_ratio=0,
)

train_loader = DataLoader(train_dataset, batch_size=4096, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=8192, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=8192, shuffle=False, num_workers=4, pin_memory=True)

print(f'Train dataset: {len(train_dataset):,} samples (positives + 1x negatives)')
print(f'Val dataset:   {len(val_dataset):,} samples')
print(f'Test dataset:  {len(test_dataset):,} samples')


## Step 4.3 — Training (Adam, lr=1e-3, BCE, early stopping patience=3)

In [ ]:
def evaluate_loader(model, loader, device, threshold=0.3):
    model.eval()
    all_labels, all_preds = [], []
    with torch.no_grad():
        for user_ids, product_ids, labels in loader:
            preds = model(user_ids.to(device), product_ids.to(device)).cpu().numpy()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.numpy().tolist())
    return np.array(all_labels), np.array(all_preds)


optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.BCELoss()

train_losses, val_f1s = [], []
best_val_f1  = 0.0
best_state   = {k: v.clone() for k, v in model.state_dict().items()}  # init so never None
patience     = 5
patience_ctr = 0

EVAL_THRESHOLD = 0.3  # lower than 0.5 — model trained on 50/50 but val is 1:16

print('Training NCF (up to 20 epochs)...')
for epoch in range(20):
    model.train()
    epoch_loss = 0.0
    for user_ids, product_ids, labels in train_loader:
        user_ids    = user_ids.to(DEVICE)
        product_ids = product_ids.to(DEVICE)
        labels      = labels.to(DEVICE)

        optimizer.zero_grad()
        preds = model(user_ids, product_ids)
        loss  = criterion(preds, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)

    labels_v, preds_v = evaluate_loader(model, val_loader, DEVICE)
    val_f1 = f1_score(labels_v, preds_v >= EVAL_THRESHOLD, zero_division=0)
    val_f1s.append(val_f1)

    print(f'Epoch {epoch + 1:02d} | Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f}')

    if val_f1 > best_val_f1:
        best_val_f1  = val_f1
        best_state   = {k: v.clone() for k, v in model.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= patience:
            print(f'Early stopping triggered at epoch {epoch + 1}')
            break

model.load_state_dict(best_state)
torch.save(best_state, f'{MODELS_DIR}/ncf_weights.pt')
print(f'Best val F1: {best_val_f1:.4f}  — model saved.')


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, color='#6366f1', linewidth=2)
ax1.set_title('Training Loss (BCE)', fontsize=13)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')

ax2.plot(val_f1s, color='#22c55e', linewidth=2)
ax2.axvline(np.argmax(val_f1s), color='red', linestyle='--',
            label=f'Best: {max(val_f1s):.4f} (epoch {np.argmax(val_f1s) + 1})')
ax2.set_title('Validation F1 Score', fontsize=13)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('F1')
ax2.legend()

plt.tight_layout()
plt.savefig(f'{OUTPUTS_DIR}/04_ncf_training.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4.4 — Evaluate on Test Set

In [ ]:
labels_t, preds_t = evaluate_loader(model, test_loader, DEVICE)
EVAL_THRESHOLD = 0.3  # match training threshold

ncf_metrics = {
    'precision': precision_score(labels_t, preds_t >= EVAL_THRESHOLD, zero_division=0),
    'recall':    recall_score(labels_t, preds_t >= EVAL_THRESHOLD, zero_division=0),
    'f1':        f1_score(labels_t, preds_t >= EVAL_THRESHOLD, zero_division=0),
    'roc_auc':   roc_auc_score(labels_t, preds_t),
}

print('NCF — Test Set Metrics')
print('-' * 40)
for k, v in ncf_metrics.items():
    print(f'  {k:<12}: {v:.4f}')

# Load and update running results
with open(f'{OUTPUTS_DIR}/model_results.json', 'r') as f:
    results = json.load(f)
results['ncf'] = ncf_metrics
with open(f'{OUTPUTS_DIR}/model_results.json', 'w') as f:
    json.dump(results, f, indent=2)

# Running comparison table
print(f'\n{"Model":<20} {"Precision":<12} {"Recall":<12} {"F1":<12} {"ROC-AUC":<12}')
print('-' * 68)
for name, m in results.items():
    auc = m.get('roc_auc', 'N/A')
    auc_str = f'{auc:.4f}' if isinstance(auc, float) else auc
    print(f'{name:<20} {m["precision"]:<12.4f} {m["recall"]:<12.4f} {m["f1"]:<12.4f} {auc_str:<12}')

## Step 4.5 — t-SNE Visualization of Product Embeddings

In [ ]:
print('Extracting product embeddings...')
product_embeddings = model.product_embedding.weight.detach().cpu().numpy()
print(f'Embedding matrix shape: {product_embeddings.shape}')

# Sample 5,000 products for t-SNE (full matrix is too large)
rng          = np.random.default_rng(42)
sample_idxs  = rng.choice(n_products, size=min(5000, n_products), replace=False)
embed_sample = product_embeddings[sample_idxs]

print('Running t-SNE (perplexity=30, 2 components)...')
tsne    = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
embed2d = tsne.fit_transform(embed_sample)
print('t-SNE complete.')

In [ ]:
# Map sampled indices back to product_ids, then look up department
idx2product = {v: k for k, v in product2idx.items()}
sampled_pids = [idx2product[i] for i in sample_idxs]

dept_lookup = (
    prior[['product_id', 'department']]
    .drop_duplicates()
    .set_index('product_id')['department']
    .to_dict()
)
depts        = [dept_lookup.get(pid, 'other') for pid in sampled_pids]
unique_depts = sorted(set(depts))

cmap   = plt.cm.get_cmap('tab20', len(unique_depts))
d2col  = {d: cmap(i) for i, d in enumerate(unique_depts)}
colors = [d2col[d] for d in depts]

fig, ax = plt.subplots(figsize=(14, 10))
for dept in unique_depts:
    mask = np.array([d == dept for d in depts])
    ax.scatter(embed2d[mask, 0], embed2d[mask, 1],
               c=[d2col[dept]], label=dept, alpha=0.6, s=8)

ax.legend(loc='upper right', fontsize=7, markerscale=3,
          ncol=2, framealpha=0.8)
ax.set_title('t-SNE of NCF Product Embeddings (colored by department)', fontsize=14)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
plt.tight_layout()
plt.savefig(f'{OUTPUTS_DIR}/04_tsne_embeddings.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

In [ ]:
# Save ID maps for app integration
with open(f'{MODELS_DIR}/ncf_id_maps.pkl', 'wb') as f:
    pickle.dump({'user2idx': user2idx, 'product2idx': product2idx}, f)

print('Saved ncf_id_maps.pkl')
print('Phase 4 complete. Proceed to 05_gru_model.ipynb')